In [3]:
import numpy as np
import pandas as pd
import os

from google.colab import drive
from pathlib import Path

In [4]:
drive_path = Path("/content/drive/MyDrive")

if not drive_path.exists():
    print("Google Drive is not mounted.")
    drive.mount("/content/drive")

project_dir = drive_path / "Nexar_Project"

print("\nGoogle Drive connected:", drive_path.exists())
print("Project directory exists:", project_dir.exists())

if project_dir.exists():
    print("\nProject folders:")

    for item in sorted(project_dir.iterdir()):
        if item.is_dir():
            print(f"[DIR]  {item.name}")
        else:
            print(f"[FILE] {item.name}")

    for folder_name in ["positive", "negative"]:

        folder = project_dir / "Dataset_2FPS" / folder_name

        print(f"\nChecking: {folder}")
        print("Directory exists:", folder.exists())

        if folder.exists():
            files = list(folder.glob("*.npz"))
            print("NPZ files found:", len(files))

else:
    print("\nNexar_Project was not found in MyDrive.")

Google Drive is not mounted.
Mounted at /content/drive

Google Drive connected: True
Project directory exists: True

Project folders:
[DIR]  Dataset_2FPS
[DIR]  Dataset_3FPS
[DIR]  Logs
[DIR]  Manifests
[DIR]  Metadata

Checking: /content/drive/MyDrive/Nexar_Project/Dataset_2FPS/positive
Directory exists: True
NPZ files found: 750

Checking: /content/drive/MyDrive/Nexar_Project/Dataset_2FPS/negative
Directory exists: True
NPZ files found: 751


In [5]:
positive_dir = Path(
    "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/positive"
)

negative_dir = Path(
    "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/negative"
)

positive_files = sorted(positive_dir.glob("*.npz"))
negative_files = sorted(negative_dir.glob("*.npz"))

print("Positive files:", len(positive_files))
print("Negative files:", len(negative_files))
print("Total files:", len(positive_files) + len(negative_files))

def inspect_npz(file_path, label):

    print(f"\n{'=' * 50}")
    print(f"{label} SAMPLE")
    print(f"File: {file_path.name}")

    try:
        with np.load(file_path, allow_pickle=False) as data:
            print("Keys:", data.files)

            for key in data.files:
                array = data[key]
                print(f"\nKey: {key}")
                print("Shape:", array.shape)
                print("Data type:", array.dtype)
                print("Size:", f"{array.nbytes / (1024**2):.2f} MB")

    except Exception as error:
        print("Error:", error)

if positive_files:
    inspect_npz(positive_files[0], "POSITIVE")

if negative_files:
    inspect_npz(negative_files[0], "NEGATIVE")

Positive files: 750
Negative files: 751
Total files: 1501

POSITIVE SAMPLE
File: 00000.npz
Keys: ['frames', 'timestamps']

Key: frames
Shape: (42, 224, 224, 3)
Data type: uint8
Size: 6.03 MB

Key: timestamps
Shape: (42,)
Data type: float32
Size: 0.00 MB

NEGATIVE SAMPLE
File: 01040.npz
Keys: ['frames', 'timestamps']

Key: frames
Shape: (79, 224, 224, 3)
Data type: uint8
Size: 11.34 MB

Key: timestamps
Shape: (79,)
Data type: float32
Size: 0.00 MB


In [8]:
positive_dir = Path(
    "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/positive"
)

negative_dir = Path(
    "/content/drive/MyDrive/Nexar_Project/Dataset_2FPS/negative"
)


def analyze_video_lengths(files, label):
    frame_counts = []
    durations = []
    timestamp_issues = []
    unreadable_files = []

    for file_path in files:

        try:
            with np.load(file_path, allow_pickle=False) as data:
                timestamps = data["timestamps"]

            frame_counts.append(len(timestamps))

            if len(timestamps) > 0:
                durations.append(
                    float(timestamps[-1] - timestamps[0])
                )

            if len(timestamps) > 1:
                if not np.all(np.diff(timestamps) > 0):
                    timestamp_issues.append(file_path.name)

        except Exception as error:
            unreadable_files.append((file_path.name, str(error)))

    if frame_counts:
        print(f"\n{'=' * 45}")
        print(f"{label} VIDEO STATISTICS")
        print(f"{'=' * 45}")

        print("Videos analyzed:", len(frame_counts))
        print("Minimum frames:", min(frame_counts))
        print("Maximum frames:", max(frame_counts))
        print("Mean frames:", round(np.mean(frame_counts), 2))
        print("Median frames:", round(np.median(frame_counts), 2))

        print("\nFrame count percentiles:")
        for p in [25, 50, 75, 90, 95, 99]:
            print(f"{p}th percentile:", round(np.percentile(frame_counts, p), 2))

        print("\nAverage timestamp duration:",
              round(np.mean(durations), 2), "seconds")

        print("Timestamp ordering issues:", len(timestamp_issues))
        print("Unreadable files:", len(unreadable_files))

        if timestamp_issues:
            print("Examples:", timestamp_issues[:5])

        if unreadable_files:
            print("Unreadable examples:", unreadable_files[:5])

    return frame_counts

positive_files = sorted(positive_dir.glob("*.npz"))
negative_files = sorted(negative_dir.glob("*.npz"))

positive_counts = analyze_video_lengths(positive_files, "POSITIVE")
negative_counts = analyze_video_lengths(negative_files, "NEGATIVE")

print("\nCLASS COMPARISON")
print("Positive videos:", len(positive_counts))
print("Negative videos:", len(negative_counts))


POSITIVE VIDEO STATISTICS
Videos analyzed: 750
Minimum frames: 7
Maximum frames: 114
Mean frames: 38.69
Median frames: 40.0

Frame count percentiles:
25th percentile: 39.0
50th percentile: 40.0
75th percentile: 41.0
90th percentile: 42.0
95th percentile: 43.0
99th percentile: 46.0

Average timestamp duration: 18.85 seconds
Timestamp ordering issues: 0
Unreadable files: 0

NEGATIVE VIDEO STATISTICS
Videos analyzed: 750
Minimum frames: 30
Maximum frames: 120
Mean frames: 74.64
Median frames: 81.0

Frame count percentiles:
25th percentile: 79.0
50th percentile: 81.0
75th percentile: 81.0
90th percentile: 82.0
95th percentile: 82.55
99th percentile: 91.0

Average timestamp duration: 36.82 seconds
Timestamp ordering issues: 0
Unreadable files: 1
Unreadable examples: [('01214_0ff59ce23a0041038f893b6b54377f5d.uploading.npz', 'File is not a zip file')]

CLASS COMPARISON
Positive videos: 750
Negative videos: 750


<div dir="rtl" style="text-align: right; line-height: 2;">

<h2>تحلیل آماری دیتاست Nexar</h2>

<h3>۱. آمار کلاس مثبت (Positive)</h3>

<ul>
<li>تعداد ویدیوهای بررسی‌شده: <b>750</b></li>
<li>حداقل تعداد فریم: <b>7</b></li>
<li>حداکثر تعداد فریم: <b>114</b></li>
<li>میانگین تعداد فریم: <b>38.69</b></li>
<li>میانه تعداد فریم: <b>40</b></li>
</ul>

<b>توزیع تعداد فریم‌ها:</b>

<ul>
<li>صدک 25 (<bdi dir="ltr">P25</bdi>): 39</li>
<li>صدک 50 (<bdi dir="ltr">P50</bdi>): 40</li>
<li>صدک 75 (<bdi dir="ltr">P75</bdi>): 41</li>
<li>صدک 90 (<bdi dir="ltr">P90</bdi>): 42</li>
<li>صدک 95 (<bdi dir="ltr">P95</bdi>): 43</li>
<li>صدک 99 (<bdi dir="ltr">P99</bdi>): 46</li>
</ul>

میانگین بازه زمانی بین اولین و آخرین فریم: <b>18.85 ثانیه</b>

<hr>

<h3>۲. آمار کلاس منفی (Negative)</h3>

<ul>
<li>تعداد ویدیوهای بررسی‌شده: <b>750</b></li>
<li>حداقل تعداد فریم: <b>30</b></li>
<li>حداکثر تعداد فریم: <b>120</b></li>
<li>میانگین تعداد فریم: <b>74.64</b></li>
<li>میانه تعداد فریم: <b>81</b></li>
</ul>

<b>توزیع تعداد فریم‌ها:</b>

<ul>
<li>صدک 25 (<bdi dir="ltr">P25</bdi>): 79</li>
<li>صدک 50 (<bdi dir="ltr">P50</bdi>): 81</li>
<li>صدک 75 (<bdi dir="ltr">P75</bdi>): 81</li>
<li>صدک 90 (<bdi dir="ltr">P90</bdi>): 82</li>
<li>صدک 95 (<bdi dir="ltr">P95</bdi>): 82.55</li>
<li>صدک 99 (<bdi dir="ltr">P99</bdi>): 91</li>
</ul>

میانگین بازه زمانی بین اولین و آخرین فریم: <b>36.82 ثانیه</b>

<hr>

<h3>۳. مقایسه آماری دو کلاس</h3>

<table dir="rtl" style="width:100%; text-align:center; border-collapse:collapse;">
<thead>
<tr>
<th style="padding:8px;">شاخص</th>
<th style="padding:8px;">Positive</th>
<th style="padding:8px;">Negative</th>
</tr>
</thead>
<tbody>
<tr><td>تعداد ویدیوها</td><td>750</td><td>750</td></tr>
<tr><td>میانگین تعداد فریم</td><td>38.69</td><td>74.64</td></tr>
<tr><td>میانه تعداد فریم</td><td>40</td><td>81</td></tr>
<tr><td>حداقل تعداد فریم</td><td>7</td><td>30</td></tr>
<tr><td>حداکثر تعداد فریم</td><td>114</td><td>120</td></tr>
<tr><td>میانگین بازه زمانی</td><td>18.85 ثانیه</td><td>36.82 ثانیه</td></tr>
</tbody>
</table>

<h3>تحلیل نتایج</h3>

<ul>
<li>تعداد نمونه‌های دو کلاس برابر است؛ بنابراین دیتاست از نظر تعداد نمونه‌ها متعادل است.</li>
<li>میانگین تعداد فریم در کلاس منفی تقریباً دو برابر کلاس مثبت است.</li>
<li>بیشتر ویدیوهای مثبت بین 39 تا 43 فریم دارند.</li>
<li>بیشتر ویدیوهای منفی بین 79 تا 82 فریم دارند.</li>
<li>علت اصلی این تفاوت، روش استخراج فریم‌هاست؛ در ویدیوهای مثبت فقط فریم‌های قبل از زمان وقوع تصادف (<bdi dir="ltr">time_of_event</bdi>) و در ویدیوهای منفی کل بازه ویدیو نمونه‌برداری شده است.</li>
</ul>

<hr>

<h3>۴. بررسی صحت داده‌ها</h3>

<table dir="rtl" style="width:100%; text-align:center; border-collapse:collapse;">
<thead>
<tr>
<th style="padding:8px;">معیار</th>
<th style="padding:8px;">Positive</th>
<th style="padding:8px;">Negative</th>
</tr>
</thead>
<tbody>
<tr><td>فایل‌های بررسی‌شده</td><td>750</td><td>750</td></tr>
<tr><td>خطای ترتیب زمانی</td><td>0</td><td>0</td></tr>
<tr><td>فایل غیرقابل‌خواندن</td><td>0</td><td>1</td></tr>
</tbody>
</table>

<ul>
<li>ترتیب زمانی <bdi dir="ltr">Timestamps</bdi> در تمام فایل‌های قابل‌خواندن صحیح بوده است.</li>
<li>یک فایل موقت با نامی شامل <bdi dir="ltr">uploading.npz</bdi> در پوشه کلاس منفی شناسایی شد.</li>
<li>این فایل هنگام خواندن با خطای <bdi dir="ltr">File is not a zip file</bdi> مواجه شد.</li>
<li>این فایل احتمالاً ناقص یا موقت است و نباید به‌عنوان نمونه معتبر وارد آموزش شود.</li>
</ul>

<hr>

<h3>۵. جمع‌بندی تحلیل آماری</h3>

<ol>
<li>دیتاست شامل <b>1500 ویدیو</b> و دو کلاس متعادل است.</li>
<li>تعداد فریم‌های ویدیوها در دو کلاس یکسان نیست.</li>
<li>بیشتر ویدیوهای مثبت بین 39 تا 43 فریم دارند.</li>
<li>بیشتر ویدیوهای منفی بین 79 تا 82 فریم دارند.</li>
<li>ترتیب زمانی <bdi dir="ltr">Timestamps</bdi> در فایل‌های قابل‌خواندن صحیح است.</li>
<li>به دلیل متغیر بودن طول ویدیوها، پیش از ورود داده‌ها به مدل <bdi dir="ltr">CNN-LSTM</bdi> باید طول دنباله‌ها مدیریت شود.</li>
</ol>

<p><b>نتیجه نهایی:</b> بررسی آماری 1500 ویدیوی دیتاست با موفقیت انجام شد و اطلاعات لازم برای طراحی روش نمونه‌برداری و آماده‌سازی ورودی مدل فراهم شده است.</p>

</div>